# Courier review analysis: walkthrough

This notebook steps through the pipeline one stage at a time so you can see what each part does.
It runs on the **synthetic demo file** by default. Change `DATA` to your Kaggle file for real results.

The same steps run in one go with:
```
python -m shipment_analysis.pipeline --input data/raw/<file>.csv --out reports/full
```

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import pandas as pd
pd.set_option("display.max_colwidth", 120)

from shipment_analysis.io import load_reviews
from shipment_analysis.clean import clean_reviews
from shipment_analysis.couriers import tag_couriers, explode_couriers
from shipment_analysis.sentiment import add_lexicon_sentiment, net_sentiment
from shipment_analysis.aspects import aspect_mentions, aspect_review_level
from shipment_analysis.topics import fit_topics
from shipment_analysis.evaluate import model_scores, confusion, disagreements

DATA = "../data/sample/demo_reviews_synthetic.csv"   # <- swap for data/raw/<kaggle file>

## 1. Load and map columns
Column names are detected automatically (`Review` → `text`, `Rating` → `rating` and so on).

In [2]:
df = load_reviews(DATA)
print(f"{len(df):,} reviews")
df.head()

900 reviews


,review_id,company,category,title,text,rating,date
0,0,a homeware store,,Terrible,Ordered dog food. Still waiting for a refund after DPD lost my order. Avoid. Box was crushed and the contents broken...,1,2025-08-28
1,1,a garden centre,,Terrible,Ordered on Monday. The DPD driver left it outside in the rain and the box was soaked. Thanks.,2,2026-07-12
2,2,a homeware store,,Terrible,Ordered some shoes. Box was crushed and the contents broken. Hermes is a joke. Hermes dumped it on the doorstep and ...,1,2025-12-10
3,3,dpd.co.uk,,Very happy,"Ordered a new kettle. Parcel arrived a day early with DPD, brilliant service. That's my experience anyway.",5,2026-09-20
4,4,an online clothes shop,,Average,Second order this month. Royal Mail delivered it. Took a couple of days. That's my experience anyway.,3,2026-04-04


## 2. Clean
Two versions of the text are kept. `text_model` keeps capitals and punctuation because VADER and the
transformer use them as signal. `text_clean` is lowercase letters only, for TF-IDF topics.

In [3]:
df = clean_reviews(df)
print(f"{len(df):,} reviews after cleaning ({df.attrs['dropped']} dropped)")
df[["text_model", "text_clean"]].head(3)

897 reviews after cleaning (3 dropped)


,text_model,text_clean
0,Terrible. Ordered dog food. Still waiting for a refund after DPD lost my order. Avoid. Box was crushed and the conte...,terrible ordered dog food still waiting for a refund after dpd lost my order avoid box was crushed and the contents ...
1,Terrible. Ordered on Monday. The DPD driver left it outside in the rain and the box was soaked. Thanks.,terrible ordered on monday the dpd driver left it outside in the rain and the box was soaked thanks
2,Terrible. Ordered some shoes. Box was crushed and the contents broken. Hermes is a joke. Hermes dumped it on the doo...,terrible ordered some shoes box was crushed and the contents broken hermes is a joke hermes dumped it on the doorste...


## 3. Which couriers are mentioned?

In [4]:
df = tag_couriers(df)
df = df[df["n_couriers"] > 0].reset_index(drop=True)
explode_couriers(df)["courier"].value_counts()

courier
Evri                234
DPD                 197
Royal Mail          179
Yodel               117
Amazon Logistics     83
DHL                  51
InPost               36
Name: count, dtype: int64

## 4. Sentiment
Stock VADER scores delivery words like *lost* or *damaged* as neutral, so a courier lexicon is added
on top (`vaderx_*`). The example below shows the difference.

In [5]:
df = add_lexicon_sentiment(df)
example = pd.DataFrame({"text_model": ["Parcel lost, then it arrived damaged and late."], "rating": [1]})
add_lexicon_sentiment(example)[["text_model", "vader_compound", "vaderx_compound"]]

,text_model,vader_compound,vaderx_compound
0,"Parcel lost, then it arrived damaged and late.",-0.6369,-0.8271


In [6]:
mentions = explode_couriers(df)
(mentions.groupby("courier")["vaderx_label"].apply(net_sentiment)
 .sort_values().round(0).rename("net sentiment"))

courier
Yodel               -5.0
Evri                -2.0
Royal Mail          20.0
Amazon Logistics    39.0
DHL                 45.0
DPD                 55.0
InPost              67.0
Name: net sentiment, dtype: float64

## 5. How well does each model agree with the star ratings?

In [7]:
model_scores(df)

,model,accuracy,macro_f1,negative_f1,n
0,VADER + courier lexicon,0.800446,0.627018,0.836795,897
1,VADER,0.769231,0.608768,0.796267,897
2,TextBlob,0.803790,0.599053,0.837681,897


In [8]:
confusion(df, 'vaderx_label')

,model: negative,model: neutral,model: positive
stars: negative,282,28,46
stars: neutral,33,13,62
stars: positive,3,7,423


Positive-scored reviews that the customer rated 1–2★. Usually sarcasm, or praise for the shop with a dig at the courier:

In [9]:
disagreements(df, 'vaderx_label', n=5)

,rating,vaderx_compound,text_model
596,1,0.8735,"Never again. Ordered dog food. Good quality, but please stop using Royal Mail. Mine was left in the bin. Hope this h..."
460,2,0.8360,"Terrible. Second order this month. Good quality, but please stop using Royal Mail. Mine was left in the bin. Hope th..."
651,2,0.8151,"Fine. First time ordering. It came via Royal Mail, the parcel was fine but tracking was a bit vague. Hope this helps..."
344,2,0.7935,"OK. Ordered on Monday. It came via Evri, the parcel was fine but tracking was a bit vague. Hope this helps someone."
616,2,0.7935,"OK. Ordered dog food. It came via DPD, the parcel was fine but tracking was a bit vague. Hope this helps someone."


## 6. Aspect-based sentiment
Each sentence is tagged with the aspects it mentions and scored on its own.

In [10]:
asp = aspect_mentions(df)
asp.sample(5, random_state=1)

,review_id,aspect,sentence,aspect_score,aspect_label
1159,584,Parcel condition,The Evri driver left it outside in the rain and the box was soaked.,0.0000,neutral
654,320,Lost / not delivered,Parcel lost by Yodel.,-0.4588,negative
764,377,Delivery speed,Evri delivered right on time and the driver was really friendly.,0.5413,positive
1399,704,Delivery speed,"Quick delivery by DPD, kept me informed the whole way.",0.2500,positive
1741,874,Delivery speed,Tracking with DPD said out for delivery for a week.,0.0000,neutral


In [11]:
asp_c = aspect_review_level(asp).merge(mentions[["review_id", "courier"]], on="review_id")
asp_c.pivot_table(index="courier", columns="aspect", values="aspect_score", aggfunc="mean").round(2)

aspect,Customer service,Delivery speed,Driver behaviour,Lost / not delivered,Parcel condition,Safe place / doorstep,Tracking & updates
courier,,,,,,,
Amazon Logistics,0.13,0.43,0.29,-0.21,0.19,-0.08,0.39
DHL,0.28,0.43,0.24,-0.41,0.15,-0.49,0.32
DPD,0.29,0.38,0.30,-0.33,0.20,-0.33,0.34
Evri,0.02,0.15,0.05,-0.30,-0.20,-0.23,0.21
InPost,0.20,0.46,0.29,-0.19,0.26,-0.16,0.35
Royal Mail,0.12,0.29,0.11,-0.33,-0.07,-0.29,0.32
Yodel,-0.18,0.10,-0.14,-0.30,-0.34,-0.23,0.26


## 7. Complaint themes (TF-IDF + NMF)

In [12]:
topics, _ = fit_topics(df.loc[df["vaderx_label"] == "negative", "text_clean"], n_topics=6)
topics

,topic,top_words,n_reviews,share
0,5,"dumped doorstep, dumped, doorstep walked, doorstep, photo knock, photo, walked photo, knock",90,0.283019
1,4,"address, wrong address, wrong, address neighbour, bring, bring round, round, neighbour",62,0.194969
2,3,"broken, box crushed, broken joke, crushed, contents, joke, crushed contents, contents broken",57,0.179245
3,1,"speak, speak person, impossible, impossible speak, tried impossible, person, lost tried, tried",40,0.125786
4,0,"arrived, arrived damaged, damaged, damaged thrown, thrown, thrown fence, fence, marked",39,0.122642
5,2,"left outside, soaked, driver left, driver, box soaked, rain, rain box, outside rain",30,0.094340


## Notes and limitations
* Star ratings are a *weak* label. On retailer pages a 1★ may be about the shop, not the courier.
* Courier detection is keyword-based, so a review that mentions two couriers counts for both.
* Aspect rules are transparent but miss phrasing they don't list. Check `aspect_sentences.csv` for misses.
* For stronger sentiment run with `--transformer` (RoBERTa), see `requirements-extra.txt`.